# Parkinson's Disease Prediction Using ANN
Dataset: `parkinsons_disease_data.csv` (2105 rows, 35 columns). Target: `Diagnosis` (0 = No, 1 = Yes).

## 0. Install Requirements (run once, then restart the kernel if anything was installed)

In [ ]:
%pip install -q tensorflow scikit-learn seaborn pandas matplotlib

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
print('TensorFlow version:', tf.__version__)

## 2. Load Dataset

In [ ]:
# Put the CSV in the same folder as this notebook, or change the path below.
FILE = 'parkinsons_disease_data.csv'

if not os.path.exists(FILE):
    print('File not found in:', os.getcwd())
    print('Files here:', os.listdir('.'))
    # On Google Colab, uncomment the next two lines to upload it:
    # from google.colab import files
    # files.upload()
    raise FileNotFoundError(f'Place {FILE} next to the notebook.')

df = pd.read_csv(FILE)
print(df.shape)
df.head()

## 3. Explore the Data

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
sns.countplot(x='Diagnosis', data=df)
plt.title('Diagnosis Distribution')
plt.show()
df['Diagnosis'].value_counts()

## 4. Preprocessing

In [ ]:
# Drop identifier / text columns (not useful for prediction)
df_model = df.drop(columns=['PatientID', 'DoctorInCharge'])
df_model.shape

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df_model.corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
X = df_model.drop('Diagnosis', axis=1).astype('float32')
y = df_model['Diagnosis'].astype('float32')
print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype('float32')
X_test = scaler.transform(X_test).astype('float32')

# Use plain numpy arrays for labels (avoids pandas index problems)
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

## 5. Build the ANN Model

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

## 6. Train the Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15,
                           restore_best_weights=True)

In [ ]:
history = model.fit(X_train, y_train,
                    validation_split=0.2,
                    epochs=100,
                    batch_size=32,
                    callbacks=[early_stop],
                    verbose=1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(history.history['loss'], label='Train')
ax[0].plot(history.history['val_loss'], label='Validation')
ax[0].set_title('Loss'); ax[0].set_xlabel('Epoch'); ax[0].legend()

ax[1].plot(history.history['accuracy'], label='Train')
ax[1].plot(history.history['val_accuracy'], label='Validation')
ax[1].set_title('Accuracy'); ax[1].set_xlabel('Epoch'); ax[1].legend()

plt.show()

## 7. Evaluate the Model

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {acc:.4f}')

In [ ]:
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=['No Parkinson', 'Parkinson']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend()
plt.show()

## 8. Predict for a Sample Patient

In [ ]:
sample = X_test[0].reshape(1, -1)
p = float(model.predict(sample, verbose=0)[0][0])
print(f'Probability of Parkinson: {p:.3f}')
print('Prediction:', 'Parkinson' if p > 0.5 else 'No Parkinson')
print('Actual:', 'Parkinson' if y_test[0] == 1 else 'No Parkinson')

## 9. Save the Model

In [ ]:
model.save('parkinsons_ann_model.keras')
print('Model saved.')

## Conclusion
An ANN with 3 hidden layers (64-32-16) was trained on clinical, lifestyle and symptom features to predict Parkinson's diagnosis. Performance was evaluated using accuracy, classification report, confusion matrix and ROC-AUC.